Заюров А.Д.

# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
#! pip install -q schedule pytest 
# установка библиотек, если ещё не

In [23]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import os
import time
import requests
import schedule
from bs4 import BeautifulSoup

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [ ]:
def get_book_data(book_url: str) -> dict:
    """
    Получает данные о книге со страницы каталога Books to Scrape.

    Загружает HTML-страницу книги, разбирает её с помощью BeautifulSoup
    и возвращает словарь с основной информацией: названием, ценой,
    рейтингом, наличием, описанием и характеристиками из таблицы
    Product Information.

    Args:
        book_url (str): URL страницы книги, например
        "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
    
    Returns:
        dict: Словарь со следующими ключами:
            - "name" (str): название книги;
            - "price" (float): цена с сайта (только число);
            - "in_stock" (int): количество книг в наличии;
            - "score" (int): рейтинг от 0 до 5
              (звёзды переводятся в число, при ошибке — 0);
            - "description" (str): текст описания книги;
            - "upc" (str): уникальный код товара;
            - "prod_type" (str): тип продукции;
            - "price_excl_tax" (float): цена без налогов;
            - "price_incl_tax" (float): цена с налогом;
            - "tax" (float): величина налога;
            - "availabl" (str): строка доступности из таблицы;
            - "num_reviews" (int): количество отзывов.
            Если какое-то поле не найдено на странице, возвращается
            значение по умолчанию из заготовки ("None", 0, 0.0 и т.д.).
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    get_dict = {
        "name": "",
        "price": 0.0,
        "in_stock": 0,
        "score": 0,
        "description": "",
        "upc": "",
        "prod_type": "",
        "price_excl_tax": 0.0,
        "price_incl_tax": 0.0,
        "tax": 0.0,
        "availabl": "",
        "num_reviews": 0
    }

    book_page = requests.get(book_url)
    book_page.encoding = "utf-8"

    soup = BeautifulSoup(book_page.text, "html.parser")

    # Поиск названия книги
    name = soup.find("h1")
    get_dict["name"] = name.string if name else "None"

    # Поиск цены книги
    price = soup.find("p", class_="price_color")
    get_dict["price"] = float(price.string.replace("£", "")) if price else 0.0

    # Поиск кол-ва книг
    in_stock = soup.find("p", class_="instock availability")
    in_stock = in_stock.get_text().replace("\n", "").strip() if in_stock else "None"
    if "In stock" in in_stock:
        get_dict["in_stock"] = int("".join(char for char in in_stock if char.isdigit()))
    else:
        get_dict["in_stock"] = 0

    # Поиск Рейтинга
    score = soup.find("p", class_="star-rating")
    score = score["class"][1] if score else "None"
    match score:
        case "One": score = 1
        case "Two": score = 2
        case "Three": score = 3
        case "Four": score = 4
        case "Five": score = 5
        case _: score = 0
    get_dict["score"] = score

    # Поиск описания
    description = soup.find("div", id = "product_description")
    get_dict["description"] = description.find_next_sibling("p").get_text().strip() if description else "None"

    # Поиск UPC
    upc = soup.find("th", string = "UPC")
    get_dict["upc"] = upc.find_next_sibling("td").get_text().strip() if upc else "None"

    # Поиск Типа продукции
    prod_type = soup.find("th", string = "Product Type")
    get_dict["prod_type"] = prod_type.find_next_sibling("td").get_text().strip() if prod_type else "None"

    # Поиск Цены без налогов
    price_excl_tax = soup.find("th", string = "Price (excl. tax)")
    get_dict["price_excl_tax"] = float(price_excl_tax.find_next_sibling("td").get_text().strip().replace("£", "")) if price_excl_tax else 0.0

    # Поиск Цены с налогом
    price_incl_tax = soup.find("th", string = "Price (incl. tax)")
    get_dict["price_incl_tax"] = float(price_incl_tax.find_next_sibling("td").get_text().strip().replace("£", "")) if price_incl_tax else 0.0

    # Поиск Цены Налогов
    tax = soup.find("th", string = "Tax")
    get_dict["tax"] = float(tax.find_next_sibling("td").get_text().strip().replace("£", "")) if tax else 0.0

    # Поиск Доступности
    availabl = soup.find("th", string = "Availability")
    get_dict["availabl"] = availabl.find_next_sibling("td").get_text().strip() if availabl else "None"

    # Поиск кол-ва обзоров
    num_reviews = soup.find("th", string = "Number of reviews")
    get_dict["num_reviews"] = int(num_reviews.find_next_sibling("td").get_text().strip()) if num_reviews else 0

    # Возвращение словаря
    return get_dict
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [6]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'name': 'A Light in the Attic',
 'price': 51.77,
 'in_stock': 22,
 'score': 3,
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up there

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [ ]:
def scrape_books(is_save: bool = False, max_pages: int = 50) -> list[dict]:
    """
    Проходит по страницам каталога Books to Scrape и собирает данные
    о всех книгах с помощью функции get_book_data.
    
    Args:
        is_save (bool): если True — сохраняет собранные данные
            в файл books_data.txt в текущей папке; иначе пропускает
            шаг сохранения.
        max_pages (int): максимальное количество страниц каталога
            для обхода (по умолчанию 50 — весь каталог).
    
    Returns:
        list[dict]: список словарей с данными по каждой книге
        (структура словаря — результат get_book_data).
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ

    # Список для хранения книг
    all_books = []

    # Цикл в котором идёт итерация по страницам котолога
    for page in range(1, max_pages + 1):
        catolog_page = requests.get(f"https://books.toscrape.com/catalogue/page-{page}.html")
        catolog_page.encoding = "utf-8"

        cat_soup = BeautifulSoup(catolog_page.text, "html.parser")

        # Поиск ссылок на книги
        cat_book_list = cat_soup.find_all("a", title = True)

        # Проверка наличия книг на странице
        if not cat_book_list:
            # Книг на странице нет
            break

        # Цикл в котором идёт итерация по книгам
        for link in cat_book_list:
            # Добавление книги в список
            all_books.append(get_book_data(f"https://books.toscrape.com/catalogue/{link.get("href")}"))

            # Доп ожидание между запросами, для уменьшения нагрузки на сайт
            time.sleep(0.25)

        # Доп ожидание между запросами, для уменьшения нагрузки на сайт
        time.sleep(0.25)

    # Сохранение собранных книг
    if is_save:
        file_path = os.path.join(os.getcwd(), "books_data.txt")
        with open(file_path, "w", encoding="utf-8") as file:
            for book in all_books:
                file.write(f"{book}\n")
        print(f"Данные сохранены в {file_path}")
    
    return all_books

    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [25]:
# Проверка работоспособности функции
res = scrape_books(is_save=True, max_pages = 1) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

Данные сохранены в c:\Users\alex-\Desktop\LabRabota\МАГИСТРАТУРА\1 Семестр\Python\ДЗ3\books_data.txt
<class 'list'> 20


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [ ]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ

def scrape_books_job() -> None:
    """
    Запускает сбор данных о книгах и сохраняет результат в файл.
    """
    
    books = scrape_books(is_save=True)
    print(f"{time.strftime('%Y-%m-%d %H:%M:%S')} — собрано книг: {len(books)}")

def daily_scheduler() -> None:
    """
    Планирует ежедневный запуск сбора данных в 19:00 и работает
    в бесконечном цикле, проверяя расписание раз в минуту.
    """

    schedule.every().day.at("19:00").do(scrape_books_job) #

    print("Планировщик запущен. Ожидаем 19:00 каждый день...")

    while True:
        schedule.run_pending()
        time.sleep(60)  # проверяем расписание раз в минуту

# КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [27]:
daily_scheduler()

Планировщик запущен. Ожидаем 19:00 каждый день...
Данные сохранены в c:\Users\alex-\Desktop\LabRabota\МАГИСТРАТУРА\1 Семестр\Python\ДЗ3\books_data.txt
2026-11-16 18:35:50 — собрано книг: 1000


KeyboardInterrupt: 

Ошибка выше вызвана прерыванием выполнения бесконечного цикла через остановку работы ячейки

## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [31]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest tests/test_scraper.py

============================= test session starts =============================
platform win32 -- Python 3.13.5, pytest-9.2.0, pluggy-1.7.0
rootdir: c:\Users\alex-\Desktop\LabRabota\МАГИСТРАТУРА\1 Семестр\Python\ДЗ3
plugins: anyio-4.9.0
collected 3 items

tests\test_scraper.py ...                                                [100%]

======================= 3 passed in 1256.19s (0:20:56) ========================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: Смог выполнить 4 задания.`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Открыл Beautiful Soup Documentation и используя его решил 1-3 задачу. В целом почти небыло проблем, однако не сразу понял функционал вида "next_element/next_sibling"`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: Использовал ИИ для корректирования описания функций, чтобы оно больше подходило под PEP-8`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: Вроде всё понятно.`

**5. Самооценка по критериям задания**:

`Ответ: 90% задания скорей всего правильно решино.`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: До этого уже парсил и единственной проблемой был поиск элементов не отмеченных специальным "id"/"class" в теге, благодоря этому заданию я об этом узнал.`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

Рефлексивная часть была написанна до исполнения задания 5 с GitHub